In [1]:
library(ggplot2)
library(dplyr)
library(Seurat)
library(tidyverse)
library(tidyr)
library(stringr)

Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘Seurat’ was built under R version 4.3.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.3.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.3.2”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘tidyverse’ was built under R version 4.3.3”
Warning message:
“package ‘readr’ was built under R version 4.3.3”
Warning message:
“package ‘stringr’ was built under R version 4.3.2”
Warning message:
“package ‘forcats’ was built under R version 4.3.3”
── Attaching core tidyverse packages ────────────────────────────────

In [ ]:
# "indir" is a custom input path, and "your_outdir" is a custom output path.
# indir=""
# your_outdir=""

In [ ]:
hyper_hypo_DHMR <- read.csv(paste0(indir,"/00-dhmr_hyper_hypo_DHMR_include_region.csv"),row.names=1,check.names=F)

In [3]:
hyper_regions <- hyper_hypo_DHMR %>%
  select(Subclass, Hyper_Region) %>%
  # Replace single quotes with double quotes
  mutate(Hyper_Region = str_replace_all(Hyper_Region, "'", "\"")) %>%
  # Converts a string to a real list
  mutate(Hyper_Region = map(Hyper_Region, ~ jsonlite::fromJSON(.))) %>%
  # Expand into separate rows
  unnest(Hyper_Region) %>%
  rename(region = Hyper_Region) %>% data.frame()

In [4]:
head(hyper_regions);dim(hyper_regions)

,Subclass,region
,<chr>,<chr>
1,L2/3 IT CTX Glut,chr1_6926313_6926536
2,L2/3 IT CTX Glut,chr1_9985135_9985403
3,L2/3 IT CTX Glut,chr1_11458287_11458492
4,L2/3 IT CTX Glut,chr1_15779422_15780221
5,L2/3 IT CTX Glut,chr1_24663290_24663767
6,L2/3 IT CTX Glut,chr1_24787577_24788300


[1] 289530      2

In [5]:
unique(hyper_regions$Subclass);length(unique(hyper_regions$Subclass))

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
 [7] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[10] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[13] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[16] "OB-out Frmd7 Gaba"     "OB Dopa-Gaba"          "OB-STR-CTX Inh IMN"   
[19] "Sncg Gaba"             "Lamp5 Gaba"            "Pvalb Gaba"           
[22] "Sst Gaba"              "STR D1 Gaba"           "STR D2 Gaba"          
[25] "ACB-BST-FS D1 Gaba"

[1] 25

In [ ]:
process_regions <- function(subclass1,subclass2) { # 'L2/3 IT CTX Glut', 'L2_3'
  df <- hyper_regions[hyper_regions$Subclass == subclass1,]
  df_split <- do.call(rbind, strsplit(df$region, "_"))
  df_split <- as.data.frame(df_split, stringsAsFactors = FALSE)
  colnames(df_split) <- c("chr", "start", "end")
  df_split$start <- as.numeric(df_split$start)
  df_split$end <- as.numeric(df_split$end)
#   df_split$mid <- (df_split$start + df_split$end) / 2
#   df_split$new_start <- pmax(0, df_split$mid - 500)
#   df_split$new_end <- df_split$mid + 500

  df_final <- data.frame(
    chr = df_split$chr,
#     start = as.integer(df_split$new_start),
#     end = as.integer(df_split$new_end),
    start = as.integer(df_split$start),
    end = as.integer(df_split$end) # ,
#     region = paste0(df_split$chr, ":", as.integer(df_split$start), "_", as.integer(df_split$end))
  )
  write.table(df_final,
              sprintf("%s/01_DHMR_hyper_bed/%s_hyper_DHMR.bed",outdir,subclass2),
              sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
  print(head(df_final))
  print(dim(df_final))
}

In [7]:
process_regions('L2/3 IT CTX Glut', 'L2_3_IT_CTX_Glut')

   chr    start      end
1 chr1  6926313  6926536
2 chr1  9985135  9985403
3 chr1 11458287 11458492
4 chr1 15779422 15780221
5 chr1 24663290 24663767
6 chr1 24787577 24788300
[1] 769   3


In [8]:
process_regions('L4/5 IT CTX Glut', 'L4_5_IT_CTX_Glut')

   chr   start     end
1 chr1 3915157 3915697
2 chr1 3989965 3990248
3 chr1 4187800 4188188
4 chr1 5155829 5156145
5 chr1 5172458 5172703
6 chr1 6926313 6926536
[1] 4351    3


In [9]:
process_regions('L5 IT CTX Glut', 'L5_IT_CTX_Glut')

   chr   start     end
1 chr1 3708364 3708588
2 chr1 3915157 3915697
3 chr1 4185804 4186540
4 chr1 4187800 4188188
5 chr1 5172458 5172703
6 chr1 5383583 5384591
[1] 3104    3


In [10]:
unique(hyper_regions$Subclass)

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
 [7] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[10] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[13] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[16] "OB-out Frmd7 Gaba"     "OB Dopa-Gaba"          "OB-STR-CTX Inh IMN"   
[19] "Sncg Gaba"             "Lamp5 Gaba"            "Pvalb Gaba"           
[22] "Sst Gaba"              "STR D1 Gaba"           "STR D2 Gaba"          
[25] "ACB-BST-FS D1 Gaba"

In [11]:
process_regions('L2/3 IT RSP Glut', 'L2_3_IT_RSP_Glut')

   chr   start     end
1 chr1 3044312 3044993
2 chr1 3344696 3344912
3 chr1 3406395 3406712
4 chr1 3985575 3985811
5 chr1 4029372 4031035
6 chr1 4077667 4077905
[1] 12658     3


In [12]:
process_regions('L4 RSP-ACA Glut', 'L4_RSP_ACA_Glut')

   chr   start     end
1 chr1 3023074 3023890
2 chr1 3156381 3156998
3 chr1 3620678 3621642
4 chr1 3676539 3677914
5 chr1 3753142 3754030
6 chr1 3819248 3819455
[1] 37313     3


In [13]:
process_regions('L5 ET CTX Glut', 'L5_ET_CTX_Glut')

   chr   start     end
1 chr1 3706655 3707199
2 chr1 3708364 3708588
3 chr1 4692632 4693073
4 chr1 4722933 4723142
5 chr1 4803135 4803693
6 chr1 5054992 5055370
[1] 13514     3


In [14]:
process_regions('SUB-ProS Glut', 'SUB_ProS_Glut')

   chr    start      end
1 chr1  3765604  3765951
2 chr1  5750151  5750443
3 chr1  6615570  6616046
4 chr1  9577282  9577559
5 chr1 17272157 17273109
6 chr1 18974516 18974786
[1] 1287    3


In [15]:
process_regions('CA1-ProS Glut', 'CA1_ProS_Glut')

   chr   start     end
1 chr1 3707593 3707888
2 chr1 3740444 3741225
3 chr1 3765604 3765951
4 chr1 3868646 3869014
5 chr1 4001106 4001597
6 chr1 4017884 4018222
[1] 6001    3


In [16]:
process_regions('CA3 Glut', 'CA3_Glut')

   chr   start     end
1 chr1 3115355 3115564
2 chr1 3151548 3152063
3 chr1 3344696 3344912
4 chr1 3765604 3765951
5 chr1 3776652 3777394
6 chr1 4031632 4032241
[1] 8760    3


In [17]:
process_regions('CLA-EPd-CTX Car3 Glut', 'CLA_EPd_CTX_Car3_Glut')

   chr   start     end
1 chr1 3162608 3163014
2 chr1 3399371 3399712
3 chr1 3764447 3764853
4 chr1 4457037 4457251
5 chr1 4472726 4473337
6 chr1 4527629 4527845
[1] 13283     3


In [18]:
process_regions('L5 NP CTX Glut', 'L5_NP_CTX_Glut')

   chr   start     end
1 chr1 4035977 4036402
2 chr1 4933630 4933992
3 chr1 4947461 4947710
4 chr1 5021102 5021386
5 chr1 6077659 6078949
6 chr1 6360954 6361211
[1] 20696     3


In [19]:
unique(hyper_regions$Subclass)

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
 [7] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[10] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[13] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[16] "OB-out Frmd7 Gaba"     "OB Dopa-Gaba"          "OB-STR-CTX Inh IMN"   
[19] "Sncg Gaba"             "Lamp5 Gaba"            "Pvalb Gaba"           
[22] "Sst Gaba"              "STR D1 Gaba"           "STR D2 Gaba"          
[25] "ACB-BST-FS D1 Gaba"

In [20]:
process_regions('L6 CT CTX Glut', 'L6_CT_CTX_Glut')

   chr   start     end
1 chr1 3908740 3909046
2 chr1 5417655 5418051
3 chr1 5958307 5958507
4 chr1 9594577 9596362
5 chr1 9624401 9624901
6 chr1 9642394 9642614
[1] 4072    3


In [21]:
process_regions('DG Glut', 'DG_Glut')

   chr    start      end
1 chr1  5012900  5013143
2 chr1  5242566  5243154
3 chr1  6862674  6863538
4 chr1  7189954  7191940
5 chr1  7358843  7359197
6 chr1 10317623 10318473
[1] 2974    3


In [22]:
process_regions('OB Eomes Ms4a15 Glut', 'OB_Eomes_Ms4a15_Glut')

   chr   start     end
1 chr1 3536190 3537098
2 chr1 4414508 4414779
3 chr1 4809258 4809481
4 chr1 6194083 6194930
5 chr1 6347599 6348467
6 chr1 6378332 6379836
[1] 9573    3


In [23]:
process_regions('OB-in Frmd7 Gaba', 'OB_in_Frmd7_Gaba')

   chr    start      end
1 chr1  3119395  3120636
2 chr1  3668344  3669309
3 chr1  9815730  9816066
4 chr1 10683939 10684288
5 chr1 10715403 10716940
6 chr1 11176388 11176643
[1] 2698    3


In [24]:
process_regions('OB-out Frmd7 Gaba', 'OB_out_Frmd7_Gaba')

   chr    start      end
1 chr1  3668344  3669309
2 chr1  5207368  5207751
3 chr1 10719951 10720376
4 chr1 15358458 15358768
5 chr1 15683498 15683848
6 chr1 17329100 17330666
[1] 5474    3


In [25]:
process_regions('OB Dopa-Gaba', 'OB_Dopa_Gaba')

   chr    start      end
1 chr1  3119395  3120636
2 chr1  3668344  3669309
3 chr1  4390596  4391145
4 chr1  9993627  9994050
5 chr1 13573976 13574336
6 chr1 15358458 15358768
[1] 5634    3


In [26]:
process_regions('OB-STR-CTX Inh IMN', 'OB_STR_CTX_Inh_IMN')

   chr    start      end
1 chr1 13299500 13299867
2 chr1 14119697 14120223
3 chr1 22526363 22526683
4 chr1 23379293 23379677
5 chr1 25179735 25180217
6 chr1 25180402 25180741
[1] 1108    3


In [27]:
unique(hyper_regions$Subclass)

[1] "L2/3 IT CTX Glut"      "L4/5 IT CTX Glut"      "L5 IT CTX Glut"       
 [4] "L2/3 IT RSP Glut"      "L4 RSP-ACA Glut"       "L5 ET CTX Glut"       
 [7] "SUB-ProS Glut"         "CA1-ProS Glut"         "CA3 Glut"             
[10] "CLA-EPd-CTX Car3 Glut" "L5 NP CTX Glut"        "L6 CT CTX Glut"       
[13] "DG Glut"               "OB Eomes Ms4a15 Glut"  "OB-in Frmd7 Gaba"     
[16] "OB-out Frmd7 Gaba"     "OB Dopa-Gaba"          "OB-STR-CTX Inh IMN"   
[19] "Sncg Gaba"             "Lamp5 Gaba"            "Pvalb Gaba"           
[22] "Sst Gaba"              "STR D1 Gaba"           "STR D2 Gaba"          
[25] "ACB-BST-FS D1 Gaba"

In [28]:
process_regions('Sncg Gaba', 'Sncg_Gaba')

   chr   start     end
1 chr1 3153311 3153607
2 chr1 3239558 3240695
3 chr1 3418966 3419937
4 chr1 3597902 3599010
5 chr1 3846253 3846622
6 chr1 3900946 3902269
[1] 19575     3


In [29]:
process_regions('Lamp5 Gaba', 'Lamp5_Gaba')

   chr   start     end
1 chr1 3019020 3019545
2 chr1 3032861 3034192
3 chr1 3076706 3077471
4 chr1 3099559 3100024
5 chr1 3115355 3115564
6 chr1 3117047 3117566
[1] 23176     3


In [30]:
process_regions('Pvalb Gaba', 'Pvalb_Gaba')

   chr   start     end
1 chr1 3076706 3077471
2 chr1 3099559 3100024
3 chr1 3115355 3115564
4 chr1 3131976 3132183
5 chr1 3153311 3153607
6 chr1 3168801 3169555
[1] 36168     3


In [31]:
process_regions('Sst Gaba', 'Sst_Gaba')

   chr   start     end
1 chr1 3019020 3019545
2 chr1 3074366 3074990
3 chr1 3076706 3077471
4 chr1 3084207 3084772
5 chr1 3090853 3092180
6 chr1 3096288 3097965
[1] 40347     3


In [32]:
process_regions('STR D1 Gaba', 'STR_D1_Gaba')

   chr   start     end
1 chr1 3119395 3120636
2 chr1 3801080 3801463
3 chr1 3900265 3900667
4 chr1 4105260 4105681
5 chr1 4950594 4950912
6 chr1 4951750 4952390
[1] 10689     3


In [33]:
process_regions('STR D2 Gaba', 'STR_D2_Gaba')

   chr    start      end
1 chr1  3119395  3120636
2 chr1  6516718  6517090
3 chr1 10719951 10720376
4 chr1 12718075 12718571
5 chr1 25106714 25107002
6 chr1 33906384 33906618
[1] 847   3


In [34]:
process_regions('ACB-BST-FS D1 Gaba', 'ACB_BST_FS_D1_Gaba')

   chr    start      end
1 chr1  3119395  3120636
2 chr1  4414508  4414779
3 chr1  5021102  5021386
4 chr1 11458287 11458492
5 chr1 11579989 11580266
6 chr1 11895854 11896072
[1] 5459    3
